# Apache Polaris — Clean All Resources

**Date:** 2026-05-15

## ⚠️ Warning
This notebook deletes ALL Polaris resources except system entities (root, service_admin).

**Deletion order matters** — must follow dependency order:
```
1. Tables & Views      (inside namespaces)
2. Namespaces          (inside catalogs)
3. Catalog Roles       (inside catalogs)
4. Catalogs
5. Principals          (except root)
6. Principal Roles     (except service_admin)
```

In [82]:
# ============================================================
# Prerequisites
# uv add requests pandas
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'

def get_token(client_id='root', client_secret='polaris-secret'):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={'grant_type': 'client_credentials', 'client_id': client_id,
              'client_secret': client_secret, 'scope': 'PRINCIPAL_ROLE:ALL'}
    )
    return r.json()['access_token']

def h(token):
    return {'Authorization': f'Bearer {token}', 'Polaris-Realm': REALM,
            'Content-Type': 'application/json'}

def delete(url, token, label):
    r = requests.delete(url, headers=h(token))
    icon = '✅' if r.status_code in [200, 204] else '⚠️ '
    print(f'  {icon} [{r.status_code}] {label}')
    return r.status_code

TOKEN = get_token()

# System entities — never delete
SKIP_PRINCIPALS     = {'root'}
SKIP_PRINCIPAL_ROLES = {'service_admin'}

print('✅ Connected as root')
print(f'   Skip principals:      {SKIP_PRINCIPALS}')
print(f'   Skip principal roles: {SKIP_PRINCIPAL_ROLES}')

✅ Connected as root
   Skip principals:      {'root'}
   Skip principal roles: {'service_admin'}


In [99]:
# ============================================================
# Preview — show what will be deleted
# ============================================================
print('=== Resources to be deleted ===\n')

catalogs       = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',       headers=h(TOKEN)).json().get('catalogs', [])
principals     = requests.get(f'{POLARIS_URL}/api/management/v1/principals',     headers=h(TOKEN)).json().get('principals', [])
principal_roles = requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', [])

print(f'Catalogs ({len(catalogs)}):')
for c in catalogs:
    cname = c['name']
    # Namespaces
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    ns_list = ns_r.json().get('namespaces', [])
    print(f'  📚 {cname}')
    # Catalog roles
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        print(f'    🔑 catalog-role: {cr["name"]}')
    for ns in ns_list:
        ns_name = ns[0] if isinstance(ns, list) else ns
        print(f'    📁 namespace: {ns_name}')
        # Tables
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            print(f'      📊 table: {t.get("name", t)}')
        # Views
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            print(f'      👁️  view: {v.get("name", v)}')

print(f'\nPrincipals ({len([p for p in principals if p["name"] not in SKIP_PRINCIPALS])} to delete):')
for p in principals:
    skip = p['name'] in SKIP_PRINCIPALS
    print(f'  {"⏭️  skip" if skip else "👤 delete"}: {p["name"]}')

print(f'\nPrincipal Roles ({len([r for r in principal_roles if r["name"] not in SKIP_PRINCIPAL_ROLES])} to delete):')
for pr in principal_roles:
    skip = pr['name'] in SKIP_PRINCIPAL_ROLES
    print(f'  {"⏭️  skip" if skip else "👥 delete"}: {pr["name"]}')

print('\n⚠️  Run next cell to execute deletion')

=== Resources to be deleted ===

Catalogs (0):

Principals (0 to delete):
  ⏭️  skip: root

Principal Roles (0 to delete):
  ⏭️  skip: service_admin

⚠️  Run next cell to execute deletion


In [100]:
# ============================================================
# Step 1 — Delete Tables and Views
# ============================================================
print('=== Step 1: Delete Tables and Views ===')

for c in catalogs:
    cname = c['name']
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # Delete tables
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            tname = t.get('name', str(t))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables/{tname}',
                TOKEN, f'table: {cname}.{ns_name}.{tname}'
            )

        # Delete views
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            vname = v.get('name', str(v))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views/{vname}',
                TOKEN, f'view:  {cname}.{ns_name}.{vname}'
            )

print('\n✅ Step 1 complete')

=== Step 1: Delete Tables and Views ===

✅ Step 1 complete


In [101]:
# ============================================================
# Dynamic Catalog Property Reinforcement & Bottom-Up Cleanup Loop
# ============================================================
print('=== Starting Scheduled Bottom-Up Resource Cleanup ===')

for c in catalogs:
    cname = c['name']
    cat_url = f'{POLARIS_URL}/api/management/v1/catalogs/{cname}'

    # ── 1. Reinforce Catalog Properties via Resilient PUT ──
    for attempt in range(3):  # Retry loop to handle 409 version conflicts dynamically
        cat_r = requests.get(cat_url, headers=h(TOKEN))
        if cat_r.status_code != 200:
            break

        cat_data = cat_r.json().get('catalog', {})
        version = cat_data.get('entityVersion', 1)
        properties = cat_data.get('properties', {})
        storage_config = cat_data.get('storageConfigInfo', {})

        # Check if the property is missing or not set to 'false'
        if properties.get('polaris.config.drop-with-purge.enabled') != 'false':
            properties['polaris.config.drop-with-purge.enabled'] = 'false'
            print(f"🔄 Setting 'drop-with-purge' to 'false' for '{cname}' (Version: {version})...")

            put_r = requests.put(
                cat_url,
                headers=h(TOKEN),
                json={
                    "currentEntityVersion": version,
                    "catalog": {
                        "name": cname,
                        "type": cat_data.get('type', 'INTERNAL'),
                        "properties": properties,
                        "storageConfigInfo": storage_config
                    }
                }
            )
            if put_r.status_code in (200, 201, 204):
                print(f"   ✅ Properties successfully updated.")
                break
            elif put_r.status_code == 409:
                print(f"   ⚠️ Version conflict (409). Fetching fresh entity version and retrying...")
                continue
            else:
                print(f"   ❌ PUT failed: [{put_r.status_code}] {put_r.text}")
                break
        else:
            print(f"   ⏭️ Catalog '{cname}' already has property configured to false.")
            break

    # ── 2. Bottom-Up Deletion Sequence ──
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # A. Delete Views First (Explicitly metadata-only)
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            vname = v.get('name', str(v))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views/{vname}?purgeRequested=false',
                TOKEN, f'view: {cname}.{ns_name}.{vname}'
            )

        # B. Delete Tables (Explicitly metadata-only)
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            tname = t.get('name', str(t))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables/{tname}?purgeRequested=false',
                TOKEN, f'table: {cname}.{ns_name}.{tname}'
            )

        # C. Delete Namespace
        delete(
            f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}',
            TOKEN, f'namespace: {cname}.{ns_name}'
        )

    # D. Delete Catalog Roles
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        crname = cr['name']
        if crname == 'catalog_admin':
            continue
        delete(
            f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}',
            TOKEN, f'catalog-role: {cname}/{crname}'
        )

    # E. Delete Catalog Entity
    delete(
        f'{POLARIS_URL}/api/management/v1/catalogs/{cname}',
        TOKEN, f'catalog: {cname}'
    )

print('\n🎉 Bottom-up cleanup execution finished!')

=== Starting Scheduled Bottom-Up Resource Cleanup ===

🎉 Bottom-up cleanup execution finished!


In [102]:
# ============================================================
# Core Resource Cleanup Sequence (Double-Escaped Metastore Bypass)
# ============================================================
print('=== Executing Double-Escaped Metastore Bypass Purge ===')

for c in catalogs:
    cname = c['name']
    cat_url = f'{POLARIS_URL}/api/management/v1/catalogs/{cname}'

    # 1. Fetch exact namespace payloads from catalog
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    namespaces = ns_r.json().get('namespaces', [])

    for ns in namespaces:
        # Convert any format to the literal leaking text string
        ns_raw_string = ns if isinstance(ns, str) else '%1F'.join(ns)

        # DOUBLE ESCAPE THE PERCENT SYMBOL: Ensure %1F goes over the wire as %251F
        # This tells the Polaris path router: "Look for the literal string, do not tokenize it."
        double_escaped_ns = ns_raw_string.replace('%1F', 'parent-ns%251Fvp-ns') if '%1F' in ns_raw_string else ns_raw_string
        if '%1F' in ns_raw_string:
            double_escaped_ns = ns_raw_string.replace('%1F', '%251F')
        else:
            double_escaped_ns = ns_raw_string.replace('\x1f', '%251F')

        print(f"💥 Targeting stuck entity: {cname}.{ns_raw_string} via token {double_escaped_ns}")

        # A. Clean tables inside the ghost namespace wrapper
        requests.delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{double_escaped_ns}/tables?purgeRequested=false', headers=h(TOKEN))
        # B. Clean views inside the ghost namespace wrapper
        requests.delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{double_escaped_ns}/views?purgeRequested=false', headers=h(TOKEN))

        # C. Delete the ghost namespace entity using the standard Iceberg endpoint
        r_del = requests.delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{double_escaped_ns}', headers=h(TOKEN))

        # D. Alternate check: Try dropping via the Management API path with double escape
        if r_del.status_code == 404:
            r_del = requests.delete(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/namespaces/{double_escaped_ns}', headers=h(TOKEN))

        print(f'   Namespace drop status: [{r_del.status_code}]')

    # 2. Finally drop the empty catalog wrapper
    r_cat = requests.delete(cat_url, headers=h(TOKEN))
    icon_cat = '✅' if r_cat.status_code in [200, 204] else '⚠️ '
    print(f'{icon_cat} [{r_cat.status_code}] catalog wrapper purge: {cname}\n')

print('🎉 Double-escaped clearance pass complete!')

=== Executing Double-Escaped Metastore Bypass Purge ===
🎉 Double-escaped clearance pass complete!


In [103]:
# ============================================================
# Core Resource Cleanup Sequence (Polaris 1.3.0 Version Override)
# ============================================================
print('=== Starting Scheduled Bottom-Up Resource Cleanup ===')

for c in catalogs:
    cname = c['name']
    cat_url = f'{POLARIS_URL}/api/management/v1/catalogs/{cname}'

    # ── 1. Check & Inject 'polaris.config.drop-with-purge.enabled' ──
    cat_r = requests.get(cat_url, headers=h(TOKEN))
    if cat_r.status_code == 200:
        cat_data = cat_r.json().get('catalog', {})
        properties = cat_data.get('properties', {}) or {}
        base_version = cat_data.get('entityVersion', 1)

        if properties.get('polaris.config.drop-with-purge.enabled') != 'false':
            properties['polaris.config.drop-with-purge.enabled'] = 'false'

            # Try the reported version first, then try incrementing if it hits a 409
            for version_to_try in [base_version, base_version + 1]:
                payload = {
                    "currentEntityVersion": version_to_try,
                    "catalog": {
                        "name": cname,
                        "type": cat_data.get('type', 'INTERNAL'),
                        "properties": properties,
                        "storageConfigInfo": cat_data.get('storageConfigInfo', {}),
                        "entityVersion": version_to_try
                    }
                }

                print(f"🔄 Injecting 'drop-with-purge=false' into catalog '{cname}' (Attempting Version: {version_to_try})...")
                put_r = requests.put(cat_url, headers=h(TOKEN), json=payload)

                if put_r.status_code in (200, 201, 204):
                    print(f"   ✅ Catalog update successful on version {version_to_try}!")
                    break
                elif put_r.status_code == 409 and version_to_try == base_version:
                    print(f"   ⚠️ Version conflict (409) on base version. Escalating counter...")
                    continue
                else:
                    print(f"   ❌ Catalog update failed: HTTP {put_r.status_code}")
        else:
            print(f"   ⏭️ Catalog '{cname}' already has property configured to false.")

    # ── 2. Follow Flow Step 2: Delete from the Bottom Up ──
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # A. Delete Views First
        v_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views', headers=h(TOKEN))
        for v in v_r.json().get('identifiers', []):
            vname = v.get('name', str(v))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views/{vname}?purgeRequested=false',
                TOKEN, f'view: {cname}.{ns_name}.{vname}'
            )

        # B. Delete Tables
        t_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables', headers=h(TOKEN))
        for t in t_r.json().get('identifiers', []):
            tname = t.get('name', str(t))
            delete(
                f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables/{tname}?purgeRequested=false',
                TOKEN, f'table: {cname}.{ns_name}.{tname}'
            )

        # C. Delete Namespace
        delete(
            f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}',
            TOKEN, f'namespace: {cname}.{ns_name}'
        )

    # D. Delete Catalog Roles
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        crname = cr['name']
        if crname == 'catalog_admin':
            continue
        delete(
            f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}',
            TOKEN, f'catalog-role: {cname}/{crname}'
        )

    # E. Delete Catalog
    delete(
        f'{POLARIS_URL}/api/management/v1/catalogs/{cname}',
        TOKEN, f'catalog: {cname}'
    )

print('\n🎉 Bottom-up cleanup execution finished!')

=== Starting Scheduled Bottom-Up Resource Cleanup ===

🎉 Bottom-up cleanup execution finished!


In [104]:
# ============================================================
# Step 2 — Delete Namespaces
# ============================================================
print('=== Step 2: Delete Namespaces ===')

for c in catalogs:
    cname = c['name']
    ns_r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(TOKEN))
    for ns in ns_r.json().get('namespaces', []):
        ns_name = ns[0] if isinstance(ns, list) else ns
        delete(
            f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}',
            TOKEN, f'namespace: {cname}.{ns_name}'
        )

print('\n✅ Step 2 complete')

=== Step 2: Delete Namespaces ===

✅ Step 2 complete


In [105]:
# ============================================================
# Step 3 — Delete Catalog Roles
# ============================================================
print('=== Step 3: Delete Catalog Roles ===')

for c in catalogs:
    cname = c['name']
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        crname = cr['name']
        if crname == 'catalog_admin':
            print(f'  ⏭️  skip: {cname}/{crname} (system role)')
            continue
        delete(
            f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}',
            TOKEN, f'catalog-role: {cname}/{crname}'
        )

print('\n✅ Step 3 complete')

=== Step 3: Delete Catalog Roles ===

✅ Step 3 complete


In [106]:
# ============================================================
# Step 4 — Delete Catalogs
# ============================================================
print('=== Step 4: Delete Catalogs ===')

for c in catalogs:
    cname = c['name']
    delete(
        f'{POLARIS_URL}/api/management/v1/catalogs/{cname}',
        TOKEN, f'catalog: {cname}'
    )

print('\n✅ Step 4 complete')

=== Step 4: Delete Catalogs ===

✅ Step 4 complete


In [107]:
# ============================================================
# Step 5 — Delete Principals
# ============================================================
print('=== Step 5: Delete Principals ===')

for p in principals:
    pname = p['name']
    if pname in SKIP_PRINCIPALS:
        print(f'  ⏭️  skip: {pname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principals/{pname}',
        TOKEN, f'principal: {pname}'
    )

print('\n✅ Step 5 complete')

=== Step 5: Delete Principals ===
  ⏭️  skip: root

✅ Step 5 complete


In [108]:
# ============================================================
# Step 6 — Delete Principal Roles
# ============================================================
print('=== Step 6: Delete Principal Roles ===')

for pr in principal_roles:
    prname = pr['name']
    if prname in SKIP_PRINCIPAL_ROLES:
        print(f'  ⏭️  skip: {prname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principal-roles/{prname}',
        TOKEN, f'principal-role: {prname}'
    )

print('\n✅ Step 6 complete')

=== Step 6: Delete Principal Roles ===
  ⏭️  skip: service_admin

✅ Step 6 complete


In [109]:
# ============================================================
# Verify — confirm everything is clean
# ============================================================
print('=== Verification ===')

remaining = {
    'catalogs':       requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',        headers=h(TOKEN)).json().get('catalogs', []),
    'principals':     requests.get(f'{POLARIS_URL}/api/management/v1/principals',      headers=h(TOKEN)).json().get('principals', []),
    'principal_roles': requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', []),
}

all_clean = True
for resource, items in remaining.items():
    names = [i['name'] for i in items]
    icon = '✅' if len(names) == 0 or all(n in SKIP_PRINCIPALS | SKIP_PRINCIPAL_ROLES for n in names) else '⚠️ '
    if icon == '⚠️ ':
        all_clean = False
    print(f'  {icon} {resource}: {names}')

print()
if all_clean:
    print('✅ All resources cleaned — only system entities remain')
else:
    print('⚠️  Some resources remain — check above')

=== Verification ===
  ✅ catalogs: []
  ✅ principals: ['root']
  ✅ principal_roles: ['service_admin']

✅ All resources cleaned — only system entities remain
